# Pseudo-Labeling & Feature Engineering - Risk Score Pipeline

Implementasi final untuk tahap Pseudo-Labeling & Feature Engineering
pada Final Project SISTECH MLOps 2026 (dataset Chicago Crimes). Rancangan ini merupakan
gabungan dari dua pendekatan yang sudah dievaluasi sebelumnya (HO1 dan Task 1), dengan
bagian terbaik dari masing-masing dipertahankan dan kekurangannya diperbaiki.

Alur pengerjaan mengikuti enam tahap berurutan:

1. Data Loading & Cleaning
2. Feature Engineering (temporal dan spasial)
3. Severity Scoring Engine
4. Validasi terhadap contoh anchor pada instruksi soal
5. Space-Time Decay (temporal dan spasial)
6. Normalisasi dan penyusunan dataset akhir (dual output)


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

pd.set_option('display.max_columns', 50)
np.random.seed(42)

In [ ]:
import gdown

FILE_ID = "12K13Az7ynV3ZVimSARImgggnbDbVqjk_"
OUTPUT  = "chicago_crimes.csv"

# Download file dari Google Drive
url = f"https://drive.google.com/uc?id={FILE_ID}"
gdown.download(url, OUTPUT, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=12K13Az7ynV3ZVimSARImgggnbDbVqjk_
From (redirected): https://drive.google.com/uc?id=12K13Az7ynV3ZVimSARImgggnbDbVqjk_&confirm=t&uuid=96fb2a70-9617-45da-924b-152527269009
To: /content/chicago_crimes.csv
100%|██████████| 2.37G/2.37G [00:18<00:00, 130MB/s]


'chicago_crimes.csv'

## Konfigurasi

Seluruh parameter pipeline dikumpulkan di satu tempat agar mudah diubah dan diaudit.
Nilai `HALF_LIFE_DAYS` dan parameter lain sudah dikalibrasi terhadap contoh perilaku
yang diberikan pada instruksi soal

In [ ]:
CONFIG = {
    'DATA_PATH': 'chicago_crimes.csv',
    'USE_SYNTHETIC_DEMO': False,

    # Bounding box kasar wilayah Kota Chicago, dipakai untuk membuang koordinat yang salah
    'BBOX': {'lat_min': 41.60, 'lat_max': 42.05, 'lon_min': -87.95, 'lon_max': -87.50},

    # Ukuran grid spasial untuk fitur agregat historis, dalam desimal derajat (2 desimal ~ 1.1 km)
    'GRID_DECIMALS': 2,

    # Half-life peluruhan temporal, dikalibrasi agar kejadian 2 tahun lalu bernilai
    # weight sekitar 0.14, sesuai contoh pada instruksi soal
    'HALF_LIFE_DAYS': 260,

    # Parameter kernel Gaussian untuk peluruhan spasial, dalam meter
    'SPATIAL_BANDWIDTH_M': 500,
    'SPATIAL_RADIUS_M': 1500,

    # Persentil untuk memotong outlier sebelum min-max scaling
    'NORMALIZE_CLIP_PERCENTILE': 99,

    'OUTPUT_DIR': 'outputs',
}

EARTH_RADIUS_M = 6371000.0

## Tahap 1: Data Loading & Cleaning

Data dimuat, kolom tanggal dan koordinat divalidasi, baris yang tidak lengkap atau
berada di luar wilayah Chicago dibuang. Retention rate dilaporkan secara eksplisit
agar proses cleaning dapat diaudit.

In [ ]:
def load_synthetic_demo(n=8000, seed=42):
    """
    Menghasilkan data sintetis dengan skema kolom yang sama seperti dataset resmi
    Chicago Crimes. Dipakai hanya untuk menguji pipeline end-to-end ketika dataset
    asli belum tersedia di environment ini. Jangan dipakai untuk submission akhir.
    """
    rng = np.random.default_rng(seed)

    primary_types = ['HOMICIDE', 'CRIM SEXUAL ASSAULT', 'KIDNAPPING', 'ROBBERY', 'ARSON', 'BATTERY',
                      'BURGLARY', 'ASSAULT', 'THEFT', 'NARCOTICS', 'CRIMINAL DAMAGE',
                      'DECEPTIVE PRACTICE', 'LIQUOR LAW VIOLATION', 'GAMBLING', 'MOTOR VEHICLE THEFT',
                      'WEAPONS VIOLATION', 'OTHER OFFENSE']
    weights = np.array([0.01, 0.01, 0.005, 0.03, 0.01, 0.12, 0.08, 0.10, 0.20, 0.08,
                         0.09, 0.05, 0.01, 0.01, 0.06, 0.03, 0.095])
    weights = weights / weights.sum()

    description_pool = ['SIMPLE', 'AGGRAVATED: HANDGUN', 'ARMED: HANDGUN', 'ARMED: KNIFE / CUTTING INSTRUMENT',
                         'DOMESTIC BATTERY SIMPLE', 'FORCIBLE ENTRY', 'ATTEMPT FORCIBLE ENTRY',
                         'FROM BUILDING', 'OVER $500', 'UNDER $500', 'TO VEHICLE', 'FROM PERSON',
                         'POSS: CANNABIS 30GMS OR LESS', 'RETAIL THEFT']

    start = pd.Timestamp('2021-01-01')
    end = pd.Timestamp('2026-06-30')
    dates = start + pd.to_timedelta(rng.integers(0, (end - start).days, size=n), unit='D') \
                   + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit='m')

    lat = rng.uniform(41.65, 42.02, size=n)
    lon = rng.uniform(-87.90, -87.55, size=n)

    return pd.DataFrame({
        'ID': np.arange(n),
        'Date': dates,
        'Primary Type': rng.choice(primary_types, size=n, p=weights),
        'Description': rng.choice(description_pool, size=n),
        'Arrest': rng.choice([True, False], size=n, p=[0.22, 0.78]),
        'Latitude': lat,
        'Longitude': lon,
    })


def load_and_clean(config):
    if config['USE_SYNTHETIC_DEMO']:
        raw = load_synthetic_demo()
    else:
        raw = pd.read_csv(config['DATA_PATH'])

    n_before = len(raw)
    df = raw.copy()

    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df[df['Date'].dt.year >= 2021]
    df = df.dropna(subset=['Date', 'Latitude', 'Longitude', 'Primary Type'])

    bbox = config['BBOX']
    df = df[
        df['Latitude'].between(bbox['lat_min'], bbox['lat_max']) &
        df['Longitude'].between(bbox['lon_min'], bbox['lon_max'])
    ]

    if 'ID' in df.columns:
        df = df.drop_duplicates(subset=['ID'])
    else:
        df = df.drop_duplicates()
    df = df.reset_index(drop=True)

    n_after = len(df)
    print(f"Baris awal            : {n_before}")
    print(f"Baris setelah cleaning: {n_after}")
    print(f"Retention rate        : {n_after / n_before * 100:.2f} persen")
    return df


df_clean = load_and_clean(CONFIG)
df_clean.head()

/tmp/ipykernel_3941/3160657498.py:50: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


Baris awal            : 8534663
Baris setelah cleaning: 1249891
Retention rate        : 14.64 persen


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,14165870,JK216131,2026-04-11,051XX S ABERDEEN ST,0820,THEFT,$500 AND UNDER,STREET,False,False,934,9.0,20.0,61.0,06,1169852.0,1870819.0,2026,04/18/2026 03:41:37 PM,41.801025,-87.652621,"(41.801025281, -87.652621265)"
1,14164088,JK214233,2026-04-11,019XX W MAYPOLE AVE,0460,BATTERY,SIMPLE,STREET,False,False,1223,12.0,27.0,28.0,08B,1163456.0,1901042.0,2026,04/18/2026 03:41:37 PM,41.884097,-87.675228,"(41.884096896, -87.675228347)"
2,14163678,JK213573,2026-04-11,027XX W BERWYN AVE,1220,DECEPTIVE PRACTICE,THEFT OF LOST / MISLAID PROPERTY,APARTMENT,True,False,2011,20.0,40.0,4.0,11,1157124.0,1935089.0,2026,04/18/2026 03:41:37 PM,41.977655,-87.697553,"(41.977655224, -87.697553085)"
3,14164937,JK215217,2026-04-11,008XX W IRVING PARK RD,0710,THEFT,THEFT FROM MOTOR VEHICLE,GAS STATION,False,False,1923,19.0,46.0,6.0,06,1169809.0,1926762.0,2026,04/18/2026 03:41:37 PM,41.954538,-87.651148,"(41.954537863, -87.65114825)"
4,14163493,JK213376,2026-04-11,039XX N CICERO AVE,0460,BATTERY,SIMPLE,BAR OR TAVERN,False,False,1634,16.0,45.0,15.0,08B,1143668.0,1925605.0,2026,04/18/2026 03:41:37 PM,41.951894,-87.747276,"(41.951893745, -87.747276287)"


## Tahap 2: Feature Engineering

Dua kelompok fitur dibentuk sebelum severity scoring diterapkan.

Fitur temporal memakai cyclical encoding (sin/cos) alih-alih encoding linear, karena
jam 23 dan jam 0 sebenarnya berdekatan secara siklis namun jauh secara nilai linear.

Fitur spasial berupa agregat historis per grid kotak sederhana (bukan untuk perhitungan
decay, hanya untuk memberi konteks tingkat aktivitas kejahatan di sekitar lokasi).

In [ ]:
def add_temporal_features(df):
    out = df.copy()
    out['hour'] = out['Date'].dt.hour
    out['dow'] = out['Date'].dt.dayofweek       # 0 = Senin, 6 = Minggu
    out['month'] = out['Date'].dt.month
    out['is_weekend'] = out['dow'].isin([5, 6]).astype(int)

    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
    out['dow_sin'] = np.sin(2 * np.pi * out['dow'] / 7)
    out['dow_cos'] = np.cos(2 * np.pi * out['dow'] / 7)
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12)
    return out


def add_spatial_grid_features(df, decimals):
    out = df.copy()
    out['grid_lat'] = out['Latitude'].round(decimals)
    out['grid_lon'] = out['Longitude'].round(decimals)
    out['grid_id'] = out['grid_lat'].astype(str) + '_' + out['grid_lon'].astype(str)

    theft_like = {'THEFT', 'MOTOR VEHICLE THEFT', 'BURGLARY'}
    narcotics_like = {'NARCOTICS'}

    grid_stats = out.groupby('grid_id').agg(
        grid_crime_count=('grid_id', 'count'),
        grid_arrest_rate=('Arrest', 'mean'),
    )
    grid_stats['grid_theft_share'] = out.assign(
        is_theft=out['Primary Type'].isin(theft_like)
    ).groupby('grid_id')['is_theft'].mean()
    grid_stats['grid_narcotics_share'] = out.assign(
        is_narcotics=out['Primary Type'].isin(narcotics_like)
    ).groupby('grid_id')['is_narcotics'].mean()

    out = out.merge(grid_stats, on='grid_id', how='left')
    return out


df_features = add_temporal_features(df_clean)
df_features = add_spatial_grid_features(df_features, CONFIG['GRID_DECIMALS'])
df_features[['grid_id', 'grid_crime_count', 'grid_arrest_rate', 'grid_theft_share', 'grid_narcotics_share']].head()

,grid_id,grid_crime_count,grid_arrest_rate,grid_theft_share,grid_narcotics_share
0,41.8_-87.65,2204,0.136116,0.220962,0.011797
1,41.88_-87.68,4971,0.110642,0.294307,0.014484
2,41.98_-87.7,1404,0.113960,0.332621,0.015670
3,41.95_-87.65,6456,0.100836,0.428903,0.005731
4,41.95_-87.75,2172,0.181860,0.434622,0.028085


## Tahap 3: Severity Scoring Engine

Skor keparahan dasar mengikuti hierarki lima tingkat yang merujuk pada pengelompokan
NIBRS/FBI. Modifier deskripsi dipisah menjadi tiga kelompok:

- `WEAPON_BONUS`: dipilih satu dengan prioritas tertinggi menang (bukan dijumlahkan),
  supaya deskripsi yang mengandung dua kata kunci senjata sekaligus tidak dihitung dobel.
- `ADDITIVE_MODIFIERS`: dijumlahkan, masing-masing menaikkan skor.
- `SUBTRACTIVE_MODIFIERS`: dijumlahkan, masing-masing menurunkan skor.

Kategori `Primary Type` yang tidak terdaftar mendapat skor default netral-rendah, bukan
diperlakukan sama seperti kategori yang sudah dikenal.

In [ ]:
SEVERITY_BASE = {
    # Tier 5 (75-100): kejahatan terhadap jiwa dan keselamatan paling berat
    'HOMICIDE': 95,
    'CRIM SEXUAL ASSAULT': 88,
    'CRIMINAL SEXUAL ASSAULT': 88,
    'KIDNAPPING': 80,

    # Tier 4 (50-74): kejahatan berat lainnya
    'ROBBERY': 55,
    'ARSON': 60,
    'BATTERY': 52,

    # Tier 3 (30-49): kejahatan tingkat sedang
    'BURGLARY': 35,
    'ASSAULT': 38,
    'THEFT': 32,
    'NARCOTICS': 40,
    'MOTOR VEHICLE THEFT': 34,
    'WEAPONS VIOLATION': 42,
    'OFFENSE INVOLVING CHILDREN': 45,

    # Tier 2 (10-29): kejahatan ringan
    'CRIMINAL DAMAGE': 20,
    'DECEPTIVE PRACTICE': 15,
    'CRIMINAL TRESPASS': 18,
    'PUBLIC PEACE VIOLATION': 12,

    # Tier 1 (0-9): pelanggaran minor
    'LIQUOR LAW VIOLATION': 5,
    'GAMBLING': 8,
}
DEFAULT_SEVERITY = 25  # fallback untuk Primary Type yang tidak terdaftar

WEAPON_BONUS = [
    (('HANDGUN', 'FIREARM'), 27),
    (('KNIFE', 'CUTTING'), 18),
    (('ARMED',), 15),
]

ADDITIVE_MODIFIERS = {
    'AGGRAVATED': 15,
    'CHILD': 15,
    'DOMESTIC': 8,
    'FORCIBLE': 10,
    'FROM PERSON': 10,
    'OVER $500': 5,
}

SUBTRACTIVE_MODIFIERS = {
    'ATTEMPT': -10,
    'UNDER $500': -5,
}


def compute_severity(primary_type, description):
    base = SEVERITY_BASE.get(str(primary_type).upper().strip(), DEFAULT_SEVERITY)
    desc = str(description).upper() if pd.notna(description) else ''

    weapon_score = 0
    for keywords, bonus in WEAPON_BONUS:
        if any(keyword in desc for keyword in keywords):
            weapon_score = bonus
            break

    additive = sum(value for keyword, value in ADDITIVE_MODIFIERS.items() if keyword in desc)
    subtractive = sum(value for keyword, value in SUBTRACTIVE_MODIFIERS.items() if keyword in desc)

    score = base + weapon_score + additive + subtractive
    return float(np.clip(score, 0, 100))


df_features['severity_score'] = df_features.apply(
    lambda row: compute_severity(row['Primary Type'], row['Description']), axis=1
)
df_features[['Primary Type', 'Description', 'severity_score']].head(10)

,Primary Type,Description,severity_score
0,THEFT,$500 AND UNDER,32.0
1,BATTERY,SIMPLE,52.0
2,DECEPTIVE PRACTICE,THEFT OF LOST / MISLAID PROPERTY,15.0
3,THEFT,THEFT FROM MOTOR VEHICLE,32.0
4,BATTERY,SIMPLE,52.0
5,CRIMINAL TRESPASS,TO LAND,18.0
6,THEFT,OVER $500,37.0
7,MOTOR VEHICLE THEFT,AUTOMOBILE,34.0
8,OTHER OFFENSE,VIOLATE ORDER OF PROTECTION,25.0
9,DECEPTIVE PRACTICE,ILLEGAL USE CASH CARD,15.0


## Tahap 4: Validasi terhadap Contoh pada Instruksi Soal

Instruksi hands-on memberikan contoh perilaku yang diharapkan secara eksplisit. Tahap ini
menjalankan pengujian langsung terhadap fungsi yang sudah didefinisikan, sebelum dipakai
pada keseluruhan dataset. Jika salah satu uji gagal, parameter di Tahap 3 dan Tahap 5 perlu
disesuaikan ulang.

In [ ]:
def run_severity_validation():
    cases = [
        ('ROBBERY', 'ARMED: HANDGUN', 82),
        ('BATTERY', 'DOMESTIC BATTERY SIMPLE', 60),
        ('THEFT', 'UNDER $500', 27),
        ('HOMICIDE', 'SIMPLE', 95),
    ]

    print('Uji Severity Scoring Engine')
    print('-' * 70)
    all_passed = True
    for primary_type, description, expected in cases:
        actual = compute_severity(primary_type, description)
        status = 'LOLOS' if abs(actual - expected) < 1e-6 else 'GAGAL'
        all_passed = all_passed and (status == 'LOLOS')
        print(f"{primary_type:12s} | {description:28s} | ekspektasi={expected:5.1f} | aktual={actual:5.1f} | {status}")
    print('-' * 70)
    print('Semua uji lolos' if all_passed else 'Ada uji yang gagal, cek parameter di atas')
    return all_passed


assert run_severity_validation(), 'Validasi severity scoring gagal, perbaiki SEVERITY_BASE atau modifier sebelum lanjut'

Uji Severity Scoring Engine
----------------------------------------------------------------------
ROBBERY      | ARMED: HANDGUN               | ekspektasi= 82.0 | aktual= 82.0 | LOLOS
BATTERY      | DOMESTIC BATTERY SIMPLE      | ekspektasi= 60.0 | aktual= 60.0 | LOLOS
THEFT        | UNDER $500                   | ekspektasi= 27.0 | aktual= 27.0 | LOLOS
HOMICIDE     | SIMPLE                       | ekspektasi= 95.0 | aktual= 95.0 | LOLOS
----------------------------------------------------------------------
Semua uji lolos


Selanjutnya perilaku peluruhan ruang-waktu diuji terhadap empat skenario yang tertulis
langsung pada instruksi soal. Jarak dan usia kejadian pada tiap skenario dipilih agar
sesuai dengan deskripsi di tabel instruksi (misalnya "kemarin" dan "2 tahun lalu").

In [ ]:
def compute_temporal_weight(event_dates, reference_date, half_life_days):
    age_days = (reference_date - event_dates).dt.total_seconds() / 86400.0
    age_days = age_days.clip(lower=0)
    return np.exp(-np.log(2) * age_days / half_life_days)


def haversine_distance_m(lat1, lon1, lat2, lon2):
    lat1_r, lon1_r, lat2_r, lon2_r = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2_r - lat1_r
    dlon = lon2_r - lon1_r
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2) ** 2
    return 2 * EARTH_RADIUS_M * np.arcsin(np.sqrt(a))


def spatial_gaussian_weight(distance_m, bandwidth_m):
    return np.exp(-(distance_m ** 2) / (2 * bandwidth_m ** 2))


def run_intuition_validation(config):
    reference_date = pd.Timestamp('2026-07-01')
    half_life = config['HALF_LIFE_DAYS']
    bandwidth = config['SPATIAL_BANDWIDTH_M']

    scenarios = [
        ('Kejahatan kemarin, grid yang sama', reference_date - pd.Timedelta(days=1), 0.0, 'sangat tinggi'),
        ('Kejahatan 2 tahun lalu, grid yang sama', reference_date - pd.Timedelta(days=730), 0.0, 'sedang (~0.14)'),
        ('Kejahatan kemarin, grid tetangga 1 km', reference_date - pd.Timedelta(days=1), 1000.0, 'rendah-sedang'),
        ('Tidak ada kejahatan dalam radius', reference_date - pd.Timedelta(days=1), 5000.0, 'mendekati nol'),
    ]

    print('Uji Perilaku Space-Time Decay')
    print('-' * 100)
    for label, event_date, distance_m, expected in scenarios:
        w_temporal = compute_temporal_weight(pd.Series([event_date]), reference_date, half_life).iloc[0]
        w_spatial = spatial_gaussian_weight(distance_m, bandwidth)
        combined = w_temporal * w_spatial
        print(f"{label:42s} | w_temporal={w_temporal:.3f} | w_spatial={w_spatial:.3f} | gabungan={combined:.3f} | ekspektasi: {expected}")
    print('-' * 100)


run_intuition_validation(CONFIG)

Uji Perilaku Space-Time Decay
----------------------------------------------------------------------------------------------------
Kejahatan kemarin, grid yang sama          | w_temporal=0.997 | w_spatial=1.000 | gabungan=0.997 | ekspektasi: sangat tinggi
Kejahatan 2 tahun lalu, grid yang sama     | w_temporal=0.143 | w_spatial=1.000 | gabungan=0.143 | ekspektasi: sedang (~0.14)
Kejahatan kemarin, grid tetangga 1 km      | w_temporal=0.997 | w_spatial=0.135 | gabungan=0.135 | ekspektasi: rendah-sedang
Tidak ada kejahatan dalam radius           | w_temporal=0.997 | w_spatial=0.000 | gabungan=0.000 | ekspektasi: mendekati nol
----------------------------------------------------------------------------------------------------


## Tahap 5: Menerapkan Space-Time Decay ke Seluruh Data

Setelah fungsi decay tervalidasi, langkah ini menghitung `temporal_weight` untuk setiap
kejadian relatif terhadap `reference_date` (tanggal terakhir pada data, bukan tanggal hari
ini, agar hasil selalu reproducible). Kolom `event_value` adalah dasar sinyal yang akan
disebarkan secara spasial pada tahap berikutnya.

In [ ]:
REFERENCE_DATE = df_features['Date'].max()

df_features['days_since'] = (REFERENCE_DATE - df_features['Date']).dt.total_seconds() / 86400.0
df_features['temporal_weight'] = compute_temporal_weight(
    df_features['Date'], REFERENCE_DATE, CONFIG['HALF_LIFE_DAYS']
)
df_features['event_value'] = df_features['severity_score'] * df_features['temporal_weight']

df_features[['Date', 'days_since', 'severity_score', 'temporal_weight', 'event_value']].head()

,Date,days_since,severity_score,temporal_weight,event_value
0,2026-04-11,0.0,32.0,1.0,32.0
1,2026-04-11,0.0,52.0,1.0,52.0
2,2026-04-11,0.0,15.0,1.0,15.0
3,2026-04-11,0.0,32.0,1.0,32.0
4,2026-04-11,0.0,52.0,1.0,52.0


## Tahap 6: Spatial Smoothing Tervektorisasi menjadi Grid-Time Lookup Table

`event_value` setiap kejadian disebarkan ke grid-grid di sekitarnya memakai kernel Gaussian
berbasis jarak riil (Haversine), bukan jarak antar sel grid. Perhitungan jarak dilakukan
lewat `BallTree` dengan metric `haversine` dan dipanggil satu kali per kombinasi
(hari dalam minggu, jam) untuk seluruh grid sekaligus, sehingga tidak diperlukan
perulangan bertingkat antara setiap pasangan grid dan kejadian.

Hasil tahap ini adalah tabel `(grid_id, dow, hour) -> risk_raw`, yang menjadi dasar untuk
endpoint REST API pada tahap berikutnya di luar notebook ini.

In [ ]:
def build_grid_reference(df):
    return (
        df[['grid_id', 'grid_lat', 'grid_lon']]
        .drop_duplicates(subset='grid_id')
        .reset_index(drop=True)
    )


def smooth_risk_to_grid(df_bucket, grid_ref, bandwidth_m, radius_m):
    if len(df_bucket) == 0:
        grid_ref = grid_ref.copy()
        grid_ref['risk_raw'] = 0.0
        return grid_ref

    event_coords_rad = np.radians(df_bucket[['Latitude', 'Longitude']].to_numpy())
    grid_coords_rad = np.radians(grid_ref[['grid_lat', 'grid_lon']].to_numpy())

    tree = BallTree(event_coords_rad, metric='haversine')
    radius_rad = radius_m / EARTH_RADIUS_M

    neighbor_indices, neighbor_distances_rad = tree.query_radius(
        grid_coords_rad, r=radius_rad, return_distance=True
    )

    event_values = df_bucket['event_value'].to_numpy()
    risk_raw = np.zeros(len(grid_ref))

    for i in range(len(grid_ref)):
        idx = neighbor_indices[i]
        if len(idx) == 0:
            continue
        dist_m = neighbor_distances_rad[i] * EARTH_RADIUS_M
        weights = spatial_gaussian_weight(dist_m, bandwidth_m)
        risk_raw[i] = np.sum(weights * event_values[idx])

    grid_ref = grid_ref.copy()
    grid_ref['risk_raw'] = risk_raw
    return grid_ref


def build_risk_lookup_table(df, config):
    grid_ref = build_grid_reference(df)
    lookup_rows = []

    for (dow, hour), df_bucket in df.groupby(['dow', 'hour']):
        grid_result = smooth_risk_to_grid(
            df_bucket, grid_ref, config['SPATIAL_BANDWIDTH_M'], config['SPATIAL_RADIUS_M']
        )
        grid_result['dow'] = dow
        grid_result['hour'] = hour
        lookup_rows.append(grid_result)

    return pd.concat(lookup_rows, ignore_index=True)


risk_lookup_raw = build_risk_lookup_table(df_features, CONFIG)
risk_lookup_raw.head()

,grid_id,grid_lat,grid_lon,risk_raw,dow,hour
0,41.8_-87.65,41.80,-87.65,509.527710,0,0
1,41.88_-87.68,41.88,-87.68,491.091600,0,0
2,41.98_-87.7,41.98,-87.70,197.337493,0,0
3,41.95_-87.65,41.95,-87.65,576.262169,0,0
4,41.95_-87.75,41.95,-87.75,149.910218,0,0


## Tahap 7: Normalisasi Risk Score ke Skala 0-100

Tiga langkah berurutan diterapkan pada `risk_raw`:

1. `log1p` untuk memampatkan distribusi yang right-skewed.
2. Clipping pada persentil 99 agar beberapa grid dengan nilai ekstrem tidak menekan
   seluruh grid lain ke ujung skala saat min-max scaling.
3. Min-max scaling ke rentang 0-100 agar mudah diinterpretasikan sebagai skor risiko.

In [ ]:
def normalize_risk(raw_series, clip_percentile):
    log_values = np.log1p(raw_series.clip(lower=0))
    upper_bound = np.percentile(log_values, clip_percentile)
    clipped = log_values.clip(upper=upper_bound)

    min_val = clipped.min()
    max_val = clipped.max()
    if max_val - min_val < 1e-9:
        return pd.Series(0.0, index=raw_series.index)

    return 100 * (clipped - min_val) / (max_val - min_val)


risk_lookup_raw['risk_score'] = normalize_risk(
    risk_lookup_raw['risk_raw'], CONFIG['NORMALIZE_CLIP_PERCENTILE']
)
risk_lookup_raw[['grid_id', 'dow', 'hour', 'risk_raw', 'risk_score']].sort_values(
    'risk_score', ascending=False
).head(10)

,grid_id,dow,hour,risk_raw,risk_score
8179,41.9_-87.63,0,11,645.771637,100.0
57,41.9_-87.72,0,0,667.815920,100.0
60,41.88_-87.75,0,0,800.404049,100.0
42,41.75_-87.61,0,0,801.217400,100.0
5,41.89_-87.64,0,0,696.004002,100.0
28,41.86_-87.72,0,0,662.509030,100.0
122666,41.75_-87.6,6,23,629.195551,100.0
75575,41.89_-87.63,4,7,790.505041,100.0
76282,41.88_-87.63,4,8,669.089763,100.0
76308,41.89_-87.63,4,8,792.618664,100.0


## Tahap 8: Menyusun Dataset Akhir (Dual Output)

Dua tabel keluaran dibentuk dari satu pipeline yang sama, sehingga keduanya selalu konsisten:

- **Dataset A** (`features_labels_events.csv`): satu baris per kejadian, menyimpan seluruh
  komponen penyusun label (`severity_score`, `days_since`, `temporal_weight`) sehingga tetap
  auditable dan siap dipakai untuk continual learning maupun drift detection.
- **Dataset B** (`risk_lookup_table.csv`): satu baris per kombinasi `(grid_id, dow, hour)`,
  siap dipanggil langsung oleh endpoint `GET /risk-score?lat=&lon=&datetime=`.

Label `risk_score` pada Dataset A diambil dari Dataset B melalui join pada `(grid_id, dow, hour)`
milik kejadian itu sendiri, agar kedua tabel tidak pernah tidak sinkron.

In [ ]:
def attach_risk_score_to_events(df_events, lookup_table):
    merged = df_events.merge(
        lookup_table[['grid_id', 'dow', 'hour', 'risk_score']],
        on=['grid_id', 'dow', 'hour'],
        how='left',
    )
    merged['risk_score'] = merged['risk_score'].fillna(0.0)
    return merged


dataset_events = attach_risk_score_to_events(df_features, risk_lookup_raw)

EVENT_COLUMNS = [
    'ID', 'Date', 'hour', 'dow', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'Latitude', 'Longitude', 'grid_id',
    'grid_crime_count', 'grid_arrest_rate', 'grid_theft_share', 'grid_narcotics_share',
    'Primary Type', 'Description',
    'severity_score', 'days_since', 'temporal_weight', 'event_value',
    'risk_score',
]
dataset_events_final = dataset_events[EVENT_COLUMNS]

LOOKUP_COLUMNS = ['grid_id', 'grid_lat', 'grid_lon', 'dow', 'hour', 'risk_raw', 'risk_score']
dataset_lookup_final = risk_lookup_raw[LOOKUP_COLUMNS]

print('Dataset A (per-event, auditable)      :', dataset_events_final.shape)
print('Dataset B (grid x dow x hour, lookup) :', dataset_lookup_final.shape)
dataset_events_final.head()

Dataset A (per-event, auditable)      : (1249891, 26)
Dataset B (grid x dow x hour, lookup) : (123144, 7)


,ID,Date,hour,dow,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,Latitude,Longitude,grid_id,grid_crime_count,grid_arrest_rate,grid_theft_share,grid_narcotics_share,Primary Type,Description,severity_score,days_since,temporal_weight,event_value,risk_score
0,14165870,2026-04-11,0,5,4,1,0.0,1.0,-0.974928,-0.222521,0.866025,-0.5,41.801025,-87.652621,41.8_-87.65,2204,0.136116,0.220962,0.011797,THEFT,$500 AND UNDER,32.0,0.0,1.0,32.0,87.518200
1,14164088,2026-04-11,0,5,4,1,0.0,1.0,-0.974928,-0.222521,0.866025,-0.5,41.884097,-87.675228,41.88_-87.68,4971,0.110642,0.294307,0.014484,BATTERY,SIMPLE,52.0,0.0,1.0,52.0,99.142584
2,14163678,2026-04-11,0,5,4,1,0.0,1.0,-0.974928,-0.222521,0.866025,-0.5,41.977655,-87.697553,41.98_-87.7,1404,0.113960,0.332621,0.015670,DECEPTIVE PRACTICE,THEFT OF LOST / MISLAID PROPERTY,15.0,0.0,1.0,15.0,78.682544
3,14164937,2026-04-11,0,5,4,1,0.0,1.0,-0.974928,-0.222521,0.866025,-0.5,41.954538,-87.651148,41.95_-87.65,6456,0.100836,0.428903,0.005731,THEFT,THEFT FROM MOTOR VEHICLE,32.0,0.0,1.0,32.0,100.000000
4,14163493,2026-04-11,0,5,4,1,0.0,1.0,-0.974928,-0.222521,0.866025,-0.5,41.951894,-87.747276,41.95_-87.75,2172,0.181860,0.434622,0.028085,BATTERY,SIMPLE,52.0,0.0,1.0,52.0,82.633185


## Tahap 9: Menyimpan Output

Kedua tabel disimpan ke direktori output dengan nama yang konsisten dengan format deliverable
pada instruksi soal.

In [ ]:
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

events_path = os.path.join(CONFIG['OUTPUT_DIR'], 'features_labels_events.csv')
lookup_path = os.path.join(CONFIG['OUTPUT_DIR'], 'risk_lookup_table.csv')

dataset_events_final.to_csv(events_path, index=False)
dataset_lookup_final.to_csv(lookup_path, index=False)

print(f"Tersimpan: {events_path}")
print(f"Tersimpan: {lookup_path}")

dataset_events_final['risk_score'].describe()

Tersimpan: outputs/features_labels_events.csv
Tersimpan: outputs/risk_lookup_table.csv


,risk_score
count,1.249891e+06
mean,8.354255e+01
std,1.237763e+01
min,9.728047e-01
25%,7.696743e+01
50%,8.545579e+01
75%,9.248397e+01
max,1.000000e+02


## Ringkasan

Pipeline di atas menghasilkan dua tabel yang konsisten satu sama lain dan sudah tervalidasi
terhadap contoh perilaku pada instruksi soal, sebelum dipakai lebih lanjut pada tahap
Model Training & Baseline Comparison, Continual Learning, dan REST API Serving.

Sebelum submission, ganti data sintetis dengan dataset Chicago Crimes asli
(`CONFIG['USE_SYNTHETIC_DEMO'] = False`, isi `CONFIG['DATA_PATH']`), lalu jalankan ulang
seluruh notebook dari awal untuk memastikan hasil akhir konsisten dengan data yang sebenarnya.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import BallTree
import joblib

df = pd.read_csv('outputs/features_labels_events.csv', parse_dates=['Date'])

FEATURE_COLUMNS = [
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'is_weekend', 'grid_crime_count', 'grid_arrest_rate',
    'grid_theft_share', 'grid_narcotics_share', 'Latitude', 'Longitude',
]
TARGET_COLUMN = 'risk_score'

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Data latih: {X_train.shape[0]} baris")
print(f"Data uji  : {X_test.shape[0]} baris")

Data latih: 999912 baris
Data uji  : 249979 baris


In [ ]:
baseline_pred = np.full(len(y_test), y_train.mean())

In [ ]:
model = RandomForestRegressor(
    n_estimators=50,
    max_depth=12,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    max_iter=100,
    max_depth=12,
    random_state=42
)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

In [ ]:
def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{label:10s} | MAE={mae:6.3f} | RMSE={rmse:6.3f} | R2={r2:6.3f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2}


print('Perbandingan Baseline vs Model')
print('-' * 55)
baseline_metrics = evaluate(y_test, baseline_pred, 'Baseline')
model_metrics = evaluate(y_test, model_pred, 'Model')

improvement_mae = (baseline_metrics['mae'] - model_metrics['mae']) / baseline_metrics['mae'] * 100
print(f"\nPerbaikan MAE terhadap baseline: {improvement_mae:.2f} persen")